# 01 FTH

            First notebook in the workflow. It loads the raw holograms, centers them,
            optionally applies the Ewald projection, defines an ROI, creates a simple
            radially symmetric Butterworth smooth beamstop mask, reconstructs the FTH
            image, and saves the nested `data` dictionary to HDF5.

            Output: `processed/Logs/data_recon_ImId_<im_id>_<user>.hdf5`.

In [ ]:
import os, sys
from os.path import join
from getpass import getuser

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
from pyFAI.detectors import Detector

def find_basefolder(start=None):
    return os.path.abspath(start or os.getcwd())

BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
sys.path.append(join(BASEFOLDER, "library", "library"))
print("Base folder:", BASEFOLDER)
import fthcore as fth
import helper_functions as helper
import interactive
from interactive import cimshow
import mask_lib
import reconstruct_rb as rec
import PETRA_MaxP04_loading as loading
import fth_phase_workflow as wf
from mask_store import MaskStore
from data_loading import SextantsNexusLoader

try:
    import cupy as cp
    import cupyx as cpx
    import CCI_core_cupy as cci
    import Phase_Retrieval as PhR
    GPU = True
    print("GPU available")
except Exception:
    import CCI_core as cci
    PhR = None
    GPU = False
    print("GPU unavailable")

%matplotlib widget
try:
    %load_ext jupyter_black
except Exception:
    pass

## Paths and experiment setup

In [ ]:
BASEFOLDER = find_basefolder()
RAW_FOLDER = "/home/experiences/sextants/com-sextants/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/"
DATAFOLDER = RAW_FOLDER
RAW_DATA_KIND = "sextants_nexus"  # "existing" or "sextants_nexus"
raw_loader = (SextantsNexusLoader(RAW_FOLDER)
              if RAW_DATA_KIND == "sextants_nexus" else None)
USER = "rb"

folder_general = helper.create_folder(join(BASEFOLDER, "processed"))
folder_logs = helper.create_folder(join(folder_general, "Logs"))

experimental_setup = {
    "ccd_dist": 0.115,
    "px_size": 20.0e-6,
    "binning": 1,
    "oversaturation": 60e3,
}
detector = Detector(
    experimental_setup["binning"] * experimental_setup["px_size"],
    experimental_setup["binning"] * experimental_setup["px_size"],
)
mnemonics = loading.mnemonics
print("Output folder:", folder_general)

## Define raw images

In [ ]:
# Edit these labels and scan IDs for the current dataset.
# The labels become data["holo"][label] groups in the HDF5 file.
hologram_inputs = {
    # mask_id=None means no precise mask. Use the frame's own ID, or any
    # other compatible image ID whose raw-coordinate mask should be reused.
    # Set stitched_file to a 00b .npz output to use a stitched raw image.
    "LV": {"id": 1266, "dark_id": 1268, "mask_id": 1266, "stitched_file": None},
    "LH": {"id": 1269, "dark_id": 1270, "mask_id": 1269, "stitched_file": None},
}

# Which labels should be used for the FTH difference hologram?
positive_label = "LH"
reference_label = "LV"
im_id = hologram_inputs[positive_label]["id"]
DATA_H5 = join(folder_logs, f"data_recon_ImId_{int(im_id):04d}_{USER}.hdf5")
print("Output HDF5:", DATA_H5)

crop = None
project_ewalds_sphere = False
ewald_method = "cubic"
heraldo = False

data = {
    "workflow": "FTH_phase_retrieval_4_notebook_sequence",
    "user": USER,
    "data_file": DATA_H5,
    "experimental_setup": experimental_setup,
    "holo": hologram_inputs,
    "hologram_labels": list(hologram_inputs.keys()),
    "positive_label": positive_label,
    "reference_label": reference_label,
    "heraldo": heraldo,
    "crop": crop,
}

## Load raw data

In [ ]:
mask_store = MaskStore(join(BASEFOLDER, "processed", "mask_pixels"))
for label, state in data["holo"].items():
    print(f"Loading {label}: image {state['id']}")
    stitched_file = state.get("stitched_file")
    stitched_mask = None
    if stitched_file:
        stitched_path = stitched_file if os.path.isabs(stitched_file) else join(BASEFOLDER, stitched_file)
        with np.load(stitched_path, allow_pickle=False) as stitched:
            image = np.asarray(stitched["image"], dtype=float)
            stitched_mask = np.asarray(stitched["mask_pixel"], dtype=np.uint8)
            state["raw_metadata"] = {"energy_eV": float(stitched["energy_eV"])}
            state["exposure"] = float(stitched["reference_exposure"])
        spe_name = stitched_path
        print(f"  stitched input: {stitched_path}")
    else:
        if raw_loader is None:
            image, spe_name = wf.load_processing(DATAFOLDER, state["id"], crop=crop)
            state["raw_metadata"] = {}
        else:
            frame = raw_loader.load(state["id"])
            image, spe_name = frame.image, str(frame.source)
            state["raw_metadata"] = dict(frame.metadata)
            state["exposure"] = frame.exposure
    if state.get("dark_id") is not None and not stitched_file:
        if raw_loader is None:
            dark, _ = wf.load_processing(DATAFOLDER, int(state["dark_id"]), crop=crop)
        else:
            dark = raw_loader.load(state["dark_id"]).image
        image = image - dark
    state["image"] = image
    state["spe_name"] = spe_name
    mask_id = state.get("mask_id")
    if stitched_mask is not None:
        state["mask_pixel_raw"] = stitched_mask
    elif mask_id is None:
        state["mask_pixel_raw"] = np.zeros(image.shape, dtype=np.uint8)
    else:
        state["mask_pixel_raw"] = mask_store.load(mask_id, image.shape)
        print(f"  raw mask from image {mask_id}: {state['mask_pixel_raw'].sum()} pixels")
    if state["mask_pixel_raw"].shape != image.shape:
        raise ValueError(f"Mask shape {state['mask_pixel_raw'].shape} != image shape {image.shape}")

first_label = next(iter(data["holo"]))
first_state = data["holo"][first_label]
if raw_loader is None:
    energy = wf.load_scan_value(
        DATAFOLDER, first_state["id"], mnemonics["energy"]
    )
else:
    energy = float(first_state["raw_metadata"]["energy_eV"])
data["experimental_setup"]["energy"] = energy
data["experimental_setup"]["lambda"] = helper.photon_energy_wavelength(
    energy, input_unit="eV"
)
print("Energy:", energy)
plot_labels = list(data["holo"].keys())
print("Plot order:", plot_labels)
cimshow(np.stack([data["holo"][label]["image"] for label in plot_labels]))

## Choose center

In [ ]:
# Find center position via widget.
pol = positive_label
c0, c1 = [622, 642]  # initial values
ic = interactive.InteractiveCenter(data["holo"][pol]["image"], c0=c0, c1=c1)

In [ ]:
# Get center positions from the widget.
center = [ic.c0, ic.c1]
print("Center:", center)
data["center"] = center
data = wf.define_centered_holograms(
    data,
    cci,
    PhR=PhR,
    project_ewalds_sphere=project_ewalds_sphere,
    ewald_method=ewald_method,
)
for label, state in data["holo"].items():
    state["mask_pixel_c"] = (
        wf.center_image(state["mask_pixel_raw"], data["center"], cci) > 0.5
    ).astype(np.uint8)
# A difference hologram is valid only where both input images are valid.
mask_pixel = np.maximum(
    data["holo"][positive_label]["mask_pixel_c"],
    data["holo"][reference_label]["mask_pixel_c"],
).astype(np.uint8)
data["mask_pixel_raw_by_label"] = {
    label: state["mask_pixel_raw"] for label, state in data["holo"].items()
}
data["mask_pixel"] = mask_pixel
plot_labels = list(data["holo"].keys())
print("Plot order:", plot_labels)
cimshow(np.stack([data["holo"][label]["image_c"] for label in plot_labels]))

## Smooth Butterworth mask and FTH reconstruction

In [ ]:
# Define the two-parameter smooth, centrosymmetric Butterworth mask here.
# mask_pixel was loaded in raw coordinates and centered after center selection.
butterworth_radius = 35
butterworth_order = 4
prop_dist = 0
phase = 0

shape = data["holo"][positive_label]["image_c"].shape
if mask_pixel.shape != shape:
    raise ValueError(f"Centered mask shape {mask_pixel.shape} != hologram shape {shape}")
mask_pixel_smooth_recipe = {
    "type": "radial_butterworth_disk",
    "radius": butterworth_radius,
    "order": butterworth_order,
}
mask_pixel_smooth = wf.butterworth_disk_mask(
    shape, butterworth_radius, butterworth_order
)
mask_multiplier = (1 - mask_pixel_smooth) * (1 - mask_pixel)

pos = np.asarray(data["holo"][positive_label]["image_c"], dtype=float)
ref = np.asarray(data["holo"][reference_label]["image_c"], dtype=float)
factor, offset = cci.dyn_factor(
    pos * (1 - mask_pixel),
    ref * (1 - mask_pixel),
    method="correlation",
    verbose=False,
    plot=False,
)
pos_scaled = pos / factor
holo_unmasked = pos_scaled - ref - offset
holo_masked = holo_unmasked * mask_multiplier
recon_unmasked = wf.fth_reconstruct(
    holo_unmasked, experimental_setup, fth, prop_dist=prop_dist, phase=phase
)
recon_masked = wf.fth_reconstruct(
    holo_masked, experimental_setup, fth, prop_dist=prop_dist, phase=phase
)

data["mask_pixel"] = mask_pixel
data["mask_pixel_smooth_recipe"] = mask_pixel_smooth_recipe
data["factor"] = factor
data["offset"] = offset
data["holo"][positive_label]["image_c_norm"] = wf.normalize_image(pos_scaled)
data["holo"][reference_label]["image_c_norm"] = wf.normalize_image(ref)
data["focus_fth"] = {
    "prop_dist": prop_dist,
    "phase": phase,
    "dx": 0,
    "dy": 0,
    "roi": None,
    "operation": "-",
}
data["fth_recipe"] = {
    "positive_label": positive_label,
    "reference_label": reference_label,
    "center": data["center"],
    "mask_pixel_smooth_recipe": mask_pixel_smooth_recipe,
    "project_ewalds_sphere": data["project_ewalds_sphere"],
    "ewald_method": ewald_method,
    "contrast": "positive / factor - reference - offset",
}

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
tmp = holo_masked
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (0.1, 99.9))
ax[0].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[0].add_patch(
    plt.Circle(
        (shape[1] / 2, shape[0] / 2),
        butterworth_radius,
        fill=False,
        edgecolor="red",
        linewidth=1.5,
    )
)
ax[0].set_title("mask * hologram")

tmp = np.real(recon_unmasked)
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[1].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[1].set_title("FTH before masking")
tmp = np.real(recon_masked)
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[2].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[2].set_title("FTH after masking")
for axis in ax:
    axis.set_axis_off()
plt.show()

## Define ROI from masked FTH reconstruction

In [ ]:
# Zoom/pan the masked FTH reconstruction, then execute the next cell to store the ROI.
fig, ax = cimshow(np.real(recon_masked))

In [ ]:
roi_s = interactive.axis_to_roi(ax)
roi = wf.slices_to_roi(roi_s)
print("ROI:", roi)

## Focus FTH reconstruction

In [ ]:
# Use the focusCDI widget to tune propagation distance and phase.
# The selected values are committed to the HDF5 data dictionary in the next cell.
focus_operation = "-"
focus_sliders = rec.focusCDI(
    data["holo"][positive_label]["image_c"] * mask_multiplier,
    data["holo"][reference_label]["image_c"] * mask_multiplier,
    roi_s,
    mask=1,
    phase=phase,
    prop_dist=prop_dist,
    experimental_setup=data["experimental_setup"],
    operation=focus_operation,
    max_prop_dist=3,
    scale=(2, 98),
)
slider_prop, slider_phase, slider_dx, slider_dy = focus_sliders[:4]

In [ ]:
# Commit selected focus values and recompute the saved FTH reconstruction.
prop_dist = slider_prop.value
phase = slider_phase.value
dx = slider_dx.value
dy = slider_dy.value

focus_fth = {
    "prop_dist": prop_dist,
    "phase": phase,
    "dx": dx,
    "dy": dy,
    "roi": roi,
    "operation": focus_operation,
}
data["focus_fth"] = focus_fth

recon_unmasked = wf.fth_reconstruct(
    holo_unmasked, experimental_setup, fth, prop_dist=prop_dist, phase=phase
)
recon_masked = wf.fth_reconstruct(
    holo_masked, experimental_setup, fth, prop_dist=prop_dist, phase=phase
)

data["recon"] = recon_masked[roi_s]
data["recon_description"] = (
    "Complex FTH reconstruction after propagation, phase shift, Butterworth masking, and ROI crop."
)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
tmp = np.real(recon_unmasked)[roi_s]
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[0].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[0].set_title("Focused FTH before masking")
tmp = np.real(recon_masked)[roi_s]
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[1].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[1].set_title("Focused FTH after masking")
for axis in ax:
    axis.set_axis_off()
plt.show()

print("prop_dist:", prop_dist)
print("phase:", phase)

im_id = data["holo"][positive_label]["id"]
png_title = f"Focused FTH after masking - im_id {im_id}"
png_name = join(folder_general, f"FTH_recon_ImId_{int(im_id):04d}_{USER}.png")
fig, ax = plt.subplots(figsize=(5, 5))
tmp = np.real(data["recon"])
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax.imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax.set_title(png_title)
ax.set_axis_off()
plt.savefig(png_name, bbox_inches="tight", transparent=False, dpi=200)
plt.show()

data["fth_png"] = png_name
print("Saved PNG:", png_name)

## Save

In [ ]:
# Always write the display PNG in the same final cell as the HDF5 result.
im_id = data["holo"][positive_label]["id"]
png_name = join(folder_general, f"FTH_recon_ImId_{int(im_id):04d}_{USER}.png")
fig, ax = plt.subplots(figsize=(5, 5))
tmp = np.real(data["recon"])
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax.imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax.set_title(f"Focused FTH after masking - im_id {im_id}")
ax.set_axis_off()
fig.savefig(png_name, bbox_inches="tight", transparent=False, dpi=200)
plt.close(fig)
data["fth_png"] = png_name
written = wf.save_data_dict(data, DATA_H5, overwrite=True)
print("Saved HDF5:", written)
print("Saved PNG:", png_name)

In [ ]:
# Workflow summary
_summary_data = data if "data" in globals() and isinstance(data, dict) else {}
_summary_h5 = globals().get("DATA_H5", _summary_data.get("data_file", "n/a"))
_summary_holo = _summary_data.get("holo", {})
_summary_pos = _summary_data.get(
    "positive_label", globals().get("positive_label", None)
)
_summary_ref = _summary_data.get(
    "reference_label", globals().get("reference_label", None)
)
_summary_im = _summary_holo.get(_summary_pos, {}).get(
    "id", globals().get("im_id", "n/a")
)
_summary_topo = _summary_holo.get(_summary_ref, {}).get(
    "id", globals().get("topo_id", "n/a")
)
print("im_id:", _summary_im)
print("topo_id:", _summary_topo)
print("HDF5:", _summary_h5)